# From Forecasts to Decisions: Inventory Planning with NumPyro and JAX

Suppose that we sell one product and that we have to decide, today, how many units to order for each of the next $15$ days. Demand is uncertain. If we order too much, the extra units sit on the shelf and cost money. If we order too little, we run out and customers wait. This is a typical inventory planning problem that many companies solve every day.

Observe that this is not a simple forecasting problem, we actually have to make a decision. These decisions are made with limited information and uncertainty, ignoring this fact is a trivialization of the problem, which usually leads to suboptimal decisions. This is where probabilistic forecasting comes in, where we do not simply analyze point estimates of future demand, but rather the entire distribution. Moreover, even with a probabilistic forecast at hand, we still have to turn it into a decision. The key is not to see this problem as a pure forecasting problem, but rather transnslate it into an optimization problem. Such optimization problems are defined by a *loss function*. In this specific case, the loss function must combine the cost of holding a stock of units (over-forecasting) and the cost of running out of stock (under-forecasting).

This problem is old and well studied, but still presenting challenges and opportunities for specific applications. There is no one-size-fits-all solution, and the best approach depends on really understanding the mechanism of the problem, and of course, the data. In this notebook, we present some techniques to solve these type of problmes through the lens of probabilistic modeling and decision theory. In particular, the ideas presented here are motivated by the following two great case studies:

- The VN2 Inventory Planning Challenge asked participants to place weekly orders for $599$ store and product pairs under shortage and holding costs. The winning entry of [Szabłowski (2026)](https://arxiv.org/abs/2601.18919) is a two-stage pipeline: a global forecasting model first, then a cost-aware order rule that targets the demand quantile implied by the costs.

-  On the optimization side, [Saunders et al. (2026)](https://www.pymc-labs.com/blog-posts/probabilistic-forecasting-optimization-under-uncertainty) at PyMC Labs show how a PyMC forecast of electricity demand enters a chance-constrained plan and a conditional value at risk (CVaR) plan in FICO Xpress, and how the two plans differ in what they control.

In order to make all concepts concrete and tangible, we study a synthetic dataset example. In this notebook, we do not want to get distracted by complex forecasting models (this is a key component, but you can find examples everywhere, in paricular in the [NumPyro Forecast Example Gallery](https://juanitorduz.github.io/numpyro_forecast/docs/examples/) 😉), but rather focus on the decision problem and the different techniques to solve it. 

We generate daily demand from a known model with a drifting level, a weekly pattern and price promotions. We fit a Bayesian forecasting model with [`numpyro_forecast`](https://juanitorduz.github.io/numpyro_forecast/) and turn its posterior predictive draws into a matrix of demand scenarios. 

From the same scenarios we build four order plans: a plan against the mean forecast, a chance-constrained plan, an expected cost plan and a CVaR plan. We solve the last two with a projected gradient method in JAX and we check the solutions against a linear program in [cvxpy](https://www.cvxpy.org/). Finally, we score the plans on held-out scenarios and on the true future. Because the data are simulated, we know the truth and we can see which plan does what.

The notebook assumes that you have written a NumPyro model before and that you know what a convex optimization problem is. It does not assume any background in inventory theory or stochastic programming.

Here is the outline:

1. **Prepare Notebook.** Magics, imports and plotting defaults.
2. **The Decision Problem.** The inventory dynamics, the loss, the costs and the four plans, with plots for the daily cost, the newsvendor trade-off and the tail risk.
3. **Generate Data.** Daily demand and a promotion calendar from a NumPyro generative model.
4. **Forecasting Model.** A local level plus weekly seasonality plus price model with a negative binomial likelihood.
5. **Model Fitting.** NUTS, diagnostics and parameter recovery.
6. **Demand Scenarios.** Joint posterior predictive draws over the horizon.
7. **Inventory Simulator.** The loss function as a `jax.lax.scan`, vectorized over scenarios and tested on the mean forecast plan.
8. **Decision Plans.** The three remaining plans and the JAX solver.
9. **Out of Sample Evaluation.** Cost and service level on held-out scenarios.
10. **Pareto Frontier.** The price of tail risk reduction.
11. **Convex Solver Cross Check.** The same problems as a linear program in cvxpy.
12. **Conclusion.** What we learned.

## Prepare Notebook

Next, we import the libraries and set the plotting defaults.

In [2]:
import warnings
from typing import NamedTuple

import arviz as az
import cvxpy as cp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import numpyro.distributions as dist
import optax
import polars as pl
import seaborn as sns
import xarray as xr
from jax import random
from jaxtyping import Array, Float, Int, PRNGKeyArray, ScalarLike
from numpyro import handlers
from numpyro.distributions import Distribution
from numpyro.infer import MCMC, NUTS, Predictive
from numpyro.infer.reparam import LocScaleReparam
from numpyro_forecast import Horizon, innovations, predict, to_datatree
from numpyro_forecast.features import fourier_features
from scipy import stats

numpyro.set_host_device_count(4)

seed: int = 42
rng_key: PRNGKeyArray = random.PRNGKey(seed)

HDI_PROB = 0.94

az.style.use("arviz-darkgrid")
az.rcParams["stats.ci_kind"] = "hdi"
az.rcParams["stats.ci_prob"] = HDI_PROB
plt.rcParams["figure.figsize"] = [10, 6]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"

%load_ext autoreload
%autoreload 2
%load_ext jaxtyping
%jaxtyping.typechecker beartype.beartype
%config InlineBackend.figure_format = "retina"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The jaxtyping extension is already loaded. To reload it, use:
  %reload_ext jaxtyping


## The Decision Problem

We plan the orders of one product for the next $15$ days. Let $H$ be the number of days in the planning horizon and let $D_t$ be the demand on day $t = 1, \dots, H$. At the start of day $t$ we receive $a_t \geq 0$ units that we ordered in advance. The vector $a = (a_1, \dots, a_H)$ is the action, the decision of this notebook. We start with $I_0$ units on hand. The inventory position $I_t$ at the end of day $t$ follows the recursion

$$
I_t = I_{t-1} + a_t - D_t.
$$

A negative $I_t$ is a backlog: demand that we could not serve on day $t$ and that we carry into the next day.

Two unit costs define the loss. Let $h$ be the holding cost of one unit that sits on the shelf at the end of a day, and let $p$ be the backorder cost of one unit that sits in backlog at the end of a day. These are the $c_h$ and $c_s$ of Szabłowski (2026), with one difference: there a shortage is a lost sale, here it is a backlog that we serve later. To write the two costs we need the positive part and the negative part of a number $u$:

$$
(u)^{+} = \max(u, 0), \qquad (u)^{-} = \max(-u, 0).
$$

The loss of the plan $a$ against the demand path $D = (D_1, \dots, D_H)$ is the sum of both costs over the horizon:

$$
L(a, D) = \sum_{t=1}^{H} \left[ h \, (I_t)^{+} + p \, (I_t)^{-} \right].
$$

Let's fix the costs and the initial stock now, since the plots below use them. We keep them in a `Problem` tuple that every function below receives. The field `alpha` is the tail probability of the chance constraint and of the CVaR, which we define in a moment. The three labels below name the tail in the figures.

In [ ]:
class Problem(NamedTuple):
    """Costs and initial state of the inventory problem."""

    holding_cost: float
    backorder_cost: float
    initial_inventory: float
    alpha: float


problem = Problem(
    holding_cost=0.1, backorder_cost=1.0, initial_inventory=50.0, alpha=0.05
)

TAIL_LABEL = f"worst {problem.alpha:.0%}"
VAR_LABEL = rf"VaR$_{{{problem.alpha:g}}}$"
CVAR_LABEL = rf"CVaR$_{{{problem.alpha:g}}}$"

Here is the daily cost as a function of the inventory position at the end of day $t$.

In [ ]:
inventory_grid = np.linspace(-100, 100, 201)
daily_cost = problem.holding_cost * np.maximum(
    inventory_grid, 0.0
) + problem.backorder_cost * np.maximum(-inventory_grid, 0.0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(inventory_grid, daily_cost, color="C0", linewidth=2)
ax.axvline(0.0, color="black", linewidth=1)
ax.annotate(
    f"slope $-p = -{problem.backorder_cost:g}$",
    xy=(-60, 60),
    xytext=(-95, 75),
    fontsize=11,
)
ax.annotate(
    f"slope $h = {problem.holding_cost:g}$", xy=(60, 6), xytext=(45, 20), fontsize=11
)
ax.set(xlabel="inventory position $I_t$", ylabel="cost on day $t$")
fig.suptitle("Daily Cost of the Inventory Position", fontsize=18, fontweight="bold");

The daily cost is zero at an inventory position of zero and grows linearly on both sides. The slope on the backlog side is $p = 1$ per unit and the slope on the shelf side is $h = 0.1$ per unit, so one unit short costs ten times as much as one unit too many. This asymmetry drives every result below.

There is no purchase cost in $L$. Is that right? The purchase bill of a plan is the unit price $c$ times the total order $A_H = \sum_t a_t$, and it does differ across plans: a plan that orders more pays more.

The bill cancels because of what happens at the end of the horizon. The units left on the shelf keep their value $c$, and the units in backlog still have to be bought after the horizon to serve the customers who wait for them. The net purchase cost of the horizon is therefore $c A_H - c I_H = c (C_H - I_0)$, where $C_H$ is the total demand of the horizon, and this number does not depend on the plan.

What a plan does control is how long units wait on the shelf, which is what $h$ charges, and how long customers wait, which is what $p$ charges. Two caveats apply. First, $h$ has to include the cost of the capital tied up in stock and any loss of value of the units while they wait. Second, if a shortage were a lost sale instead of a backlog, the cancellation would fail, and $p$ would have to include the margin of the lost sale.

The forecasting model gives us the posterior predictive distribution of the demand path $D$. We represent it by $S$ joint draws $D^{(1)}, \dots, D^{(S)}$, one path of $H$ days each, which we call scenarios. The plan $a$ must be fixed before we observe any demand, so one plan has to hold up across all of these scenarios. We approximate the expected loss of a plan by its sample average over the scenarios:

$$
\text{E}[L(a, D)] \approx \frac{1}{S} \sum_{s=1}^{S} L(a, D^{(s)}).
$$

The recursion for $I_t$ is a running sum, so we can unroll it. Let $A_t = \sum_{\tau \leq t} a_\tau$ be the cumulative orders and $C_t = \sum_{\tau \leq t} D_\tau$ the cumulative demand up to day $t$. Then

$$
I_t = I_0 + A_t - C_t.
$$

In other words, the inventory on day $t$ is the gap between the cumulative order path and the cumulative demand path, shifted by the initial stock. The inventory depends on the plan only through the running total $A_t$ and on the demand only through the running total $C_t$. Many statements below are simpler in these cumulative terms: a plan is a non-decreasing path $A_t$ that we place above or below the uncertain path $C_t - I_0$.

We compare four ways of turning the scenarios into a plan.

### Mean Forecast Plan

We replace the distribution by the mean path of the scenarios, $\bar{D} = \frac{1}{S} \sum_{s} D^{(s)}$, and solve the deterministic problem

$$
\min_{a \geq 0} \; L(a, \bar{D}).
$$

This is what a point forecast gives you. Against a single path, the cheapest plan keeps the inventory at zero, so the cumulative orders follow the cumulative mean demand.

### Chance-Constrained Plan

We ask for a service level instead of a cost. Let $\alpha$ be the stockout probability we accept on any given day. The constraint is

$$
\text{P}(I_t \geq 0) \geq 1 - \alpha \quad \text{for every } t = 1, \dots, H,
$$

and we take the cheapest plan that satisfies it. Note that the constraint holds day by day. It says that on any single day the chance of a backlog is at most $\alpha$. It does not say that the chance of at least one backlog somewhere in the $15$ days is at most $\alpha$.

That probability is larger, because the plan has $15$ chances to fail. How much larger depends on how the days move together. If the days were independent, the probability would be $1 - (1 - \alpha)^{15} \approx 54\%$ for $\alpha = 0.05$, and the union bound caps it at $15 \alpha = 75\%$ in any case. Positive dependence across days, which a shared demand level creates, lowers it.

We report both numbers in the evaluation: the stockout probability on a given day, which the constraint controls, and the probability of at least one stockout in the horizon, which it does not.

Let $Q_{1-\alpha}(C_t)$ be the $(1 - \alpha)$ quantile of the cumulative demand $C_t$ across the scenarios. With $I_t = I_0 + A_t - C_t$ the constraint becomes

$$
A_t \geq Q_{1-\alpha}(C_t) - I_0.
$$

The uncertainty collapses into a quantile path and no optimizer is needed. Note that the constraint counts how often we run out and ignores by how much.

### Expected Cost Plan

We minimize the sample average of the loss,

$$
\min_{a \geq 0} \; \frac{1}{S} \sum_{s=1}^{S} L(a, D^{(s)}),
$$

which is the sample average approximation of the Bayes decision. Here is an important observation. In cumulative terms the loss is a sum over days, and the term for day $t$ depends on the plan only through $A_t$:

$$
L(a, D) = \sum_{t=1}^{H} \left[ h \, (I_0 + A_t - C_t)^{+} + p \, (C_t - I_0 - A_t)^{+} \right].
$$

Each term is the cost of holding a stock of $I_0 + A_t$ units against an uncertain demand $C_t$, with $h$ per unit left over and $p$ per unit short. This single-period problem has a name, the newsvendor problem, after the vendor who buys newspapers in the morning without knowing how many will sell.

Its solution has a simple form. Suppose we raise the stock by one unit. With probability $\text{P}(C_t \leq I_0 + A_t)$ the unit is left over and costs $h$, and with the remaining probability it covers a unit of demand and saves $p$. The expected cost stops decreasing when the two balance:

$$
h \, \text{P}(C_t \leq I_0 + A_t) = p \, \left[1 - \text{P}(C_t \leq I_0 + A_t)\right]
\quad \Longleftrightarrow \quad
\text{P}(C_t \leq I_0 + A_t) = \frac{p}{p + h}.
$$

The optimal stock is the $p / (p + h)$ quantile of the demand distribution, and the ratio $p / (p + h)$ is called the critical fractile. The plan that minimizes the expected loss therefore sets $I_0 + A_t$ to the $p / (p + h)$ quantile of the cumulative demand $C_t$ on every day. Let's compute the fractile for our costs and plot the expected cost of a single period as a function of the stock, for a Normal demand with mean $60$ and standard deviation $15$.

In [ ]:
critical_fractile = problem.backorder_cost / (
    problem.backorder_cost + problem.holding_cost
)

demand_mean, demand_sd = 60.0, 15.0
stock_grid = np.linspace(20, 120, 201)
z = (stock_grid - demand_mean) / demand_sd
# closed forms of E[(q - D)^+] and E[(D - q)^+] for a Normal demand
expected_leftover = demand_sd * (z * stats.norm.cdf(z) + stats.norm.pdf(z))
expected_shortage = demand_sd * (stats.norm.pdf(z) - z * (1 - stats.norm.cdf(z)))
holding_part = problem.holding_cost * expected_leftover
backorder_part = problem.backorder_cost * expected_shortage
optimal_stock = demand_mean + demand_sd * stats.norm.ppf(critical_fractile)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    stock_grid,
    holding_part + backorder_part,
    color="C0",
    linewidth=2,
    label="expected cost",
)
ax.plot(stock_grid, holding_part, color="C2", linestyle=":", label="holding part")
ax.plot(stock_grid, backorder_part, color="C3", linestyle=":", label="backorder part")
ax.axvline(demand_mean, color="gray", linewidth=1, label="mean demand")
ax.axvline(
    optimal_stock,
    color="C1",
    linewidth=2,
    label=f"{critical_fractile:.3f} quantile (critical fractile)",
)
ax.legend(loc="upper right")
ax.set(xlabel="stock $q$", ylabel="expected cost of one period")
fig.suptitle("The Newsvendor Trade-Off", fontsize=18, fontweight="bold");

The expected cost has its minimum at the $0.909$ quantile, about $80$ units, and not at the mean of $60$. The backorder part makes the curve steep on the left and the holding part makes it flat on the right, so a stock above the mean is cheaper than the mean itself.

Here are two remarks. First, the constraint $a \geq 0$ means that $A_t$ is non-decreasing. Cumulative demand is non-decreasing in every scenario, so its quantile path is non-decreasing too, and the constraint does not bind.

Second, this plan accepts a stockout on about $9\%$ of the days. The service level is a consequence of the costs, and the plan has no service target of its own. The same fractile explains why the smallest feasible chance-constrained plan is also the cheapest one: a service level of $1 - \alpha = 0.95$ is above the fractile, so ordering more than the quantile path adds more expected holding cost than it saves in backorder cost.

This is also the order rule of the winning VN2 entry. Szabłowski (2026) computes the fractile $c_s / (c_s + c_h)$ from the competition costs, maps it to a Normal quantile around a point forecast with a calibrated spread, and orders up to that level after projecting the inventory through the lead time. The report names the limitation of the Normal step:

> In production, $q^\star$ would be implemented via empirical quantiles or a probabilistic forecast, avoiding any normality assumption.

That is what we do here: we read the quantile from the scenarios of the probabilistic forecast.

### CVaR Plan

We put a price on the tail. Let $\text{VaR}_\alpha(L) = Q_{1-\alpha}(L)$ be the $(1 - \alpha)$ quantile of the loss, the value at risk. The conditional value at risk is the expected loss beyond that quantile:

$$
\text{CVaR}_\alpha(L) = \text{E}\left[ L \mid L \geq \text{VaR}_\alpha(L) \right].
$$

Here is how we estimate it from the scenarios. We compute the loss of the plan in every scenario, which gives $S$ numbers, and we sort them from largest to smallest. Let $L_{(1)} \geq L_{(2)} \geq \dots \geq L_{(S)}$ denote the sorted losses, so that $L_{(1)}$ is the largest loss and $L_{(S)}$ the smallest. Let $k = \alpha S$ be an integer. The sample estimate of the conditional value at risk is the mean of the $k$ largest losses:

$$
\widehat{\text{CVaR}}_\alpha(L) = \frac{1}{k} \sum_{i=1}^{k} L_{(i)}.
$$

This equals the definition above as long as no two losses tie at the quantile, which we do not expect with real-valued losses. Let's draw the two quantities on a skewed loss density, a LogNormal with $\alpha = 0.05$. Both quantities have closed forms for this density, which we use for the lines.

In [ ]:
log_mean, log_sd = 5.0, 0.6
loss_grid = np.linspace(0, 1_000, 1_001)
loss_density = stats.lognorm.pdf(loss_grid, s=log_sd, scale=np.exp(log_mean))
z_alpha = stats.norm.ppf(1 - problem.alpha)
expected_loss_example = np.exp(log_mean + log_sd**2 / 2)
var_example = np.exp(log_mean + log_sd * z_alpha)
cvar_example = expected_loss_example * stats.norm.cdf(log_sd - z_alpha) / problem.alpha

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(loss_grid, loss_density, color="C0", linewidth=2, label="loss density")
ax.fill_between(
    loss_grid,
    loss_density,
    where=loss_grid >= var_example,
    color="C3",
    alpha=0.3,
    label=f"{TAIL_LABEL} of outcomes",
)
ax.axvline(expected_loss_example, color="gray", linewidth=1, label="expected loss")
ax.axvline(var_example, color="C1", linewidth=2, label=VAR_LABEL)
ax.axvline(cvar_example, color="C3", linewidth=2, label=CVAR_LABEL)
ax.legend(loc="upper right")
ax.set(xlabel="loss", ylabel="density", yticks=[])
fig.suptitle(
    "Value at Risk and Conditional Value at Risk", fontsize=18, fontweight="bold"
);

The value at risk is the point that the loss exceeds with probability $5\%$. The conditional value at risk is the average loss in the shaded region beyond it, so it sits further to the right. It is well above the expected loss because the density has a long right tail.

We use the same $\alpha = 0.05$ for the CVaR tail and for the chance constraint. We minimize

$$
\min_{a \geq 0} \; \text{E}[L(a, D)] + \lambda \, \text{CVaR}_\alpha(L(a, D)),
$$

where $\lambda \geq 0$ is the weight we put on the worst $\alpha$ fraction of the outcomes. With $\lambda = 0$ we recover the expected cost plan. The CVaR term counts how bad the worst outcomes are, not how often they happen.

All plans use the same scenarios. The differences we see below come from the rule alone. Let's generate the data.

## Generate Data

We simulate daily demand from a generative model with a slowly drifting level, a weekly pattern, price promotions and negative binomial noise. We write the model in NumPyro, fix the true parameters with the `do` handler and take one draw with `Predictive`.

Let's fix the length of the history, the horizon and the promotion settings in a `Setup` tuple. We keep one year of daily history ($52$ weeks) and a horizon of $15$ days.

In [ ]:
class Setup(NamedTuple):
    """Length of the history and the horizon, and the promotion settings."""

    t_obs: int = 364  # one year of daily history
    horizon: int = 15  # days to plan
    discount: float = 0.3  # price cut in a promotion week
    promo_prob: float = 0.1  # chance that a given week of the history is a promotion

    @property
    def duration(self) -> int:
        return self.t_obs + self.horizon


setup = Setup()

Why price? A covariate helps the forecast only if we know its future values. Price qualifies because the retailer sets it, so the promotion calendar for the horizon is part of the plan. Weather or competitor actions would need a forecast of their own.

A promotion lasts one calendar week and cuts the list price by $30\%$. In the history, each week is a promotion week with probability $0.1$. In the horizon the calendar is the retailer's plan: the second week of the horizon (days $8$ to $14$) is a promotion week, and we pass it to the generator as an input.

Here is the data generating process. Let $\delta_t$ be the daily increment of the level, $\ell_t$ the level, $f(t)$ the vector of Fourier features of the weekly cycle, $r_t$ the price relative to the list price and $\mu_t$ the expected demand on day $t$:

$$
\begin{aligned}
\delta_t &\sim \text{Normal}(0, \sigma_\delta), \qquad \ell_t = \sum_{\tau \leq t} \delta_\tau, \\
\log \mu_t &= \beta_0 + \ell_t + f(t)^\top \beta + \gamma \log r_t, \\
D_t &\sim \text{NegativeBinomial}(\mu_t, \phi),
\end{aligned}
$$

where $\beta_0$ is the `intercept`, $\beta$ the Fourier coefficients `beta`, $\gamma$ the price `elasticity`, $\sigma_\delta$ the `drift_scale` and $\phi$ the `concentration` of the negative binomial (a larger $\phi$ is closer to a Poisson). The price enters as a log, and this is what makes $\gamma$ an elasticity. Taking exponentials,

$$
\mu_t = \exp\left(\beta_0 + \ell_t + f(t)^\top \beta\right) \cdot r_t^{\gamma},
$$

so a relative price $r_t$ multiplies the demand by $r_t^{\gamma}$. In a promotion week $r_t = 0.7$, and with $\gamma = -1.5$ the demand is multiplied by $0.7^{-1.5} \approx 1.71$.

First, let's build the Fourier design matrix over the full duration (history plus horizon) and define the generator, together with the true parameter values that we intervene on. The promotion weeks of the history are Bernoulli draws under a `week` plate, repeated over the seven days of each week. The `log_price` site is deterministic so that we can read the price path back from the draw.

In [ ]:
fourier = fourier_features(setup.duration, period=7.0, num_terms=3)
# the retailer's plan for the horizon: a promotion in its second week
promo_future = jnp.zeros(setup.horizon).at[7:14].set(1.0)


def demand_generator(
    fourier: Float[Array, "duration features"],
    promo_future: Float[Array, " horizon"],
    setup: Setup,
) -> None:
    """Daily demand with a drifting level, a weekly pattern and price promotions."""
    duration, num_features = fourier.shape
    intercept = numpyro.sample("intercept", dist.Normal(4.0, 1.0))
    beta = numpyro.sample(
        "beta", dist.Normal(0.0, 0.3).expand([num_features]).to_event(1)
    )
    elasticity = numpyro.sample("elasticity", dist.Normal(0.0, 1.0))
    drift_scale = numpyro.sample("drift_scale", dist.HalfNormal(0.05))
    concentration = numpyro.sample("concentration", dist.Gamma(20.0, 1.0))

    # one Bernoulli per week of history, repeated over its seven days
    with numpyro.plate("week", setup.t_obs // 7):
        promo_week = numpyro.sample("promo_week", dist.Bernoulli(setup.promo_prob))

    promo = jnp.concatenate([jnp.repeat(promo_week, 7), promo_future])
    log_price = numpyro.deterministic(
        "log_price", promo * jnp.log(1.0 - setup.discount)
    )

    with numpyro.plate("time", duration):
        drift = numpyro.sample("drift", dist.Normal(0.0, drift_scale))
        level = jnp.cumsum(drift)  # random walk level
        log_rate = intercept + level + fourier @ beta + elasticity * log_price
        numpyro.sample("y", dist.NegativeBinomial2(jnp.exp(log_rate), concentration))


true_params = {
    "intercept": jnp.log(60.0),
    "beta": jnp.array([0.15, 0.05, -0.08, 0.10, 0.03, -0.02]),
    "elasticity": jnp.array(-1.5),
    "drift_scale": jnp.array(0.01),
    "concentration": jnp.array(30.0),
}

Next, we take one draw from the intervened model. We keep the demand and the price path, and we split the demand into the observed history and the true future. The covariate matrix of the forecasting model is the Fourier design plus the log relative price as a last column. The negative binomial draws are integers already, and we cast the true future to floats once because the inventory simulator works in floats.

In [ ]:
rng_key, key_data = random.split(rng_key)
generator = Predictive(handlers.do(demand_generator, data=true_params), num_samples=1)
draw = generator(key_data, fourier, promo_future, setup)

demand_full = draw["y"][0]
log_price = draw["log_price"][0]
covariates = jnp.concatenate([fourier, log_price[:, None]], axis=1)

# time at axis -2, one observation dim at -1
data = demand_full[: setup.t_obs, None]
demand_future_true = demand_full[setup.t_obs :].astype(jnp.float32)
days = np.arange(setup.duration)
days_future = days[setup.t_obs :]
promo_days_future = days_future[np.asarray(promo_future) > 0]

Let's look at the demand series and the price path below it. The dashed line marks the start of the planning horizon.

In [ ]:
fig, axes = plt.subplots(
    nrows=2, figsize=(12, 7), sharex=True, height_ratios=[3, 1], layout="constrained"
)
ax = axes[0]
ax.plot(
    days[: setup.t_obs],
    np.asarray(data[:, 0]),
    color="C0",
    linewidth=1,
    label="Observed history",
)
ax.plot(
    days_future,
    np.asarray(demand_future_true),
    "o-",
    markersize=3,
    color="C1",
    label="True future (held out)",
)
ax.axvline(setup.t_obs - 0.5, color="black", linestyle="--", linewidth=1)
ax.legend(loc="upper left")
ax.set(ylabel="units", ylim=(0, 1.2 * float(demand_full.max())))
ax = axes[1]
ax.step(days, np.exp(np.asarray(log_price)), where="mid", color="C3")
ax.axvline(setup.t_obs - 0.5, color="black", linestyle="--", linewidth=1)
ax.set(xlabel="day", ylabel="price / list price", ylim=(0.6, 1.05))
fig.suptitle("Daily Demand and Price", fontsize=18, fontweight="bold");

We see a weekly cycle on top of a level that drifts over the year, and a block of high demand in every promotion week. The blocks are the part of the series that the price covariate explains. The true future appears in the plots as a reference, and we score the plans on it at the very end.

## Forecasting Model

The forecasting model has the same structure as the generator, so it is well specified. We write it as a `numpyro_forecast` model: a plain function `(covariates, data=None)` whose first line derives the train/forecast split from the array shapes.

- **Horizon.** `Horizon.from_data` reads the observed length from `data` and the full duration from `covariates`. The horizon is whatever the covariates have beyond the data.
- **Innovations.** `innovations` samples the level increments under a `time` plate over the history and under a separate `time_future` plate over the horizon, so one model both trains and forecasts.
- **Prediction.** `predict` with a link function registers `obs` over the training window and `forecast` over the horizon, here with a negative binomial likelihood on the log rate.

Let's define the model. The last column of `covariates` is the log relative price. The prior on `elasticity` is centered at zero, so the data determine its sign and size.

In [ ]:
def demand_model(
    covariates: Float[Array, "duration features"],
    data: Int[Array, "t_obs 1"] | None = None,
) -> None:
    """Local level plus weekly seasonality plus price, negative binomial likelihood."""
    horizon = Horizon.from_data(covariates, data)
    fourier, log_price = covariates[..., :-1], covariates[..., -1]
    num_features = fourier.shape[-1]
    intercept = numpyro.sample("intercept", dist.Normal(4.0, 1.0))
    beta = numpyro.sample(
        "beta", dist.Normal(0.0, 0.3).expand([num_features]).to_event(1)
    )
    elasticity = numpyro.sample("elasticity", dist.Normal(0.0, 1.0))
    drift_scale = numpyro.sample("drift_scale", dist.HalfNormal(0.05))
    concentration = numpyro.sample("concentration", dist.Gamma(20.0, 1.0))
    # non-centered increments: the sampler sees standardized drifts
    drift = innovations(
        horizon, "drift", dist.Normal(0.0, drift_scale), reparam=LocScaleReparam(0)
    )
    level = jnp.cumsum(drift, axis=-2)
    regression = (fourier @ beta + elasticity * log_price)[..., None]
    log_rate = intercept + level + regression

    def likelihood(eta: Float[Array, "..."]) -> Distribution:
        return dist.NegativeBinomial2(jnp.exp(eta), concentration)

    predict(horizon, likelihood, log_rate)

Let's render the model on the training window to see its structure.

In [ ]:
numpyro.render_model(
    demand_model,
    model_args=(covariates[: setup.t_obs], data),
    render_distributions=True,
    render_params=True,
)

The global parameters sit outside the `time` plate. Inside it we see `drift_decentered`, the standardized increments that the `LocScaleReparam` samples, and the deterministic `drift` that rescales them by `drift_scale`. The observation site `obs` collects the whole training window. On the training window there is no horizon, so the `time_future` plate does not appear.

## Model Fitting

We fit the model with NUTS, four chains of $1{,}000$ draws each. Every draw is one joint scenario later on, so we keep all $4{,}000$ of them.

In [ ]:
rng_key, key_fit = random.split(rng_key)
mcmc = MCMC(
    NUTS(demand_model, target_accept_prob=0.98),
    num_warmup=1_000,
    num_samples=1_000,
    num_chains=4,
    progress_bar=False,
)
mcmc.run(key_fit, covariates[: setup.t_obs], data)
posterior = mcmc.get_samples()

`to_datatree` turns the posterior into an `xarray.DataTree` in the ArviZ schema. Because `covariates` extends beyond `data`, the tree also carries in-sample posterior predictive draws and the forecast draws over the horizon.

In [ ]:
rng_key, key_tree = random.split(rng_key)
idata = to_datatree(key_tree, demand_model, posterior, data, covariates, num_chains=4)
idata

The tree holds the posterior, the in-sample posterior predictive, the observed data and the forecast draws as separate groups, so every ArviZ function works on it directly. Let's check the sampler diagnostics for the global parameters.

In [ ]:
global_params = ["intercept", "beta", "elasticity", "drift_scale", "concentration"]
az.summary(idata, var_names=global_params)

The $\hat{R}$ values are at $1.0$ and the effective sample sizes are above $1{,}000$ for every parameter, which is good! Let's also look at the traces.

In [ ]:
pc = az.plot_trace_dist(idata, var_names=global_params)
pc.viz["figure"].item().suptitle("Posterior Traces", fontsize=18, fontweight="bold");

The chains mix well. Next, we check that the posterior recovers the parameters we fixed in the generator. The vertical lines are the true values.

In [ ]:
true_values = xr.Dataset(
    {
        "intercept": float(true_params["intercept"]),
        "beta": (idata["posterior"]["beta"].dims[-1], np.asarray(true_params["beta"])),
        "elasticity": float(true_params["elasticity"]),
        "drift_scale": float(true_params["drift_scale"]),
        "concentration": float(true_params["concentration"]),
    }
)

pc = az.plot_dist(
    idata,
    var_names=global_params,
    kind="kde",
    visuals={
        "point_estimate": False,
        "point_estimate_text": False,
        "credible_interval": False,
    },
)
az.add_lines(
    pc,
    values=true_values,
    aes_by_visuals={"ref_line": []},
    visuals={
        "ref_line": {"color": "C3", "linestyle": "-", "linewidth": 2, "zorder": 2}
    },
)
pc.viz["figure"].item().suptitle(
    "Posterior Against True Parameters", fontsize=18, fontweight="bold"
);

The true values sit inside the posterior mass for every parameter. The `elasticity` is identified from the promotion weeks in the history, which is what makes the price covariate useful over the horizon. Next, let's compare the in-sample posterior predictive distribution with the observed series over the last $90$ days of history. We pack the draws into a small tree and plot it with `az.plot_lm`, with the $50\%$ and $94\%$ HDI bands. `plot_lm` draws the inner band opaque, so we add a small helper that lightens both bands and keeps the mean line visible.

In [ ]:
def lighten_bands(pc: az.PlotCollection, x: str) -> None:
    """Set the alpha of the 50% and 94% HDI bands of a `plot_lm` collection."""
    bands = pc.viz["ci_band"][x]

    for prob, alpha in [(0.5, 0.5), (HDI_PROB, 0.25)]:
        for band in bands.sel(prob=prob).values.flat:
            band.set_alpha(alpha)


last_quarter = slice(setup.t_obs - 90, setup.t_obs)
ppc_tree = az.from_dict(
    {
        "posterior_predictive": {
            "obs": idata["posterior_predictive"]["obs"]
            .sel(obs_dim=0)
            .values[:, :, last_quarter]
        },
        "observed_data": {"obs": np.asarray(data[last_quarter, 0])},
        "constant_data": {"day": days[last_quarter].astype(float)},
    },
    coords={"time": days[last_quarter]},
    dims={"obs": ["time"], "day": ["time"]},
)

pc = az.plot_lm(
    ppc_tree,
    x="day",
    y="obs",
    y_obs="obs",
    smooth=False,
    ci_prob=(0.5, HDI_PROB),
    point_estimate="mean",
    visuals={
        "ci_band": {"color": "C0"},
        "pe_line": {"color": "C0", "alpha": 1.0},
        "observed_scatter": {"color": "black", "alpha": 1.0, "s": 12},
    },
    figure_kwargs={"figsize": (12, 6)},
)
lighten_bands(pc, "day")
ax = pc.viz["plot"]["day"].item()
pc.viz["ci_band"]["day"].sel(prob=HDI_PROB).item().set_label(f"{HDI_PROB:.0%} HDI")
pc.viz["ci_band"]["day"].sel(prob=0.5).item().set_label("50% HDI")
pc.viz["pe_line"]["day"].item().set_label("posterior predictive mean")
pc.viz["observed_scatter"]["day"].item().set_label("observed")
ax.legend(loc="upper left")
ax.set(xlabel="day", ylabel="units", title="")
pc.viz["figure"].item().suptitle(
    "Posterior Predictive Check", fontsize=18, fontweight="bold"
);

The band covers $86$ of the $90$ observations, close to the nominal $94\%$, and the mean tracks both the weekly cycle and the promotion week. The model is a reasonable description of the history, so let's move to the horizon.

## Demand Scenarios

`to_datatree` stores the forecast draws as `obs` in the `predictions` group, shaped `(chain, draw, time, obs_dim)`. We flatten the sample dimensions into a scenario matrix of shape $(S, H)$, where each row is one *joint* draw of the $15$ days. We then shuffle the rows and split them. The first $1{,}000$ scenarios are the optimization set that every plan sees. We hold out the remaining $3{,}000$ and use them only to score the plans.

A note on day numbers. The plots keep the day index of the series, so the horizon runs from day $364$ to day $378$. In the text we count the days of the horizon from $1$ to $15$, so day $1$ of the horizon is day $364$ on the plots and the promotion week, days $8$ to $14$ of the horizon, is days $371$ to $377$.

In [ ]:
N_SCENARIOS_OPT = 1_000

scenarios = jnp.asarray(
    idata["predictions"]["obs"].values.reshape(-1, setup.horizon), dtype=jnp.float32
)
rng_key, key_split = random.split(rng_key)
scenarios = scenarios[random.permutation(key_split, scenarios.shape[0])]
demand_opt, demand_eval = scenarios[:N_SCENARIOS_OPT], scenarios[N_SCENARIOS_OPT:]
cumulative_demand_opt = jnp.cumsum(demand_opt, axis=1)

Let's plot the forecast as a fan over the last $60$ days of history. The shaded column is the promotion week in the horizon.

In [ ]:
def shade_promotion(ax: plt.Axes) -> None:
    """Shade the promotion days of the horizon."""
    ax.axvspan(
        promo_days_future[0] - 0.5,
        promo_days_future[-1] + 0.5,
        color="C3",
        alpha=0.15,
        label="Promotion week",
    )


forecast_tree = az.from_dict(
    {
        "posterior_predictive": {
            "obs": idata["predictions"]["obs"].sel(obs_dim=0).values
        },
        "observed_data": {"obs": np.asarray(demand_future_true)},
        "constant_data": {"day": days_future.astype(float)},
    },
    coords={"time": days_future},
    dims={"obs": ["time"], "day": ["time"]},
)

pc = az.plot_lm(
    forecast_tree,
    x="day",
    y="obs",
    y_obs="obs",
    smooth=False,
    ci_prob=(0.5, HDI_PROB),
    point_estimate="mean",
    visuals={
        "ci_band": {"color": "C2"},
        "pe_line": {"color": "C2", "alpha": 1.0, "linewidth": 2},
        "observed_scatter": {"color": "C1", "alpha": 1.0, "s": 20},
    },
    figure_kwargs={"figsize": (12, 6)},
)
lighten_bands(pc, "day")
ax = pc.viz["plot"]["day"].item()
pc.viz["ci_band"]["day"].sel(prob=HDI_PROB).item().set_label(f"{HDI_PROB:.0%} HDI")
pc.viz["ci_band"]["day"].sel(prob=0.5).item().set_label("50% HDI")
pc.viz["pe_line"]["day"].item().set_label("Forecast mean")
pc.viz["observed_scatter"]["day"].item().set_label("True future")
last_two_months = slice(setup.t_obs - 60, setup.t_obs)
ax.plot(
    days[last_two_months],
    np.asarray(data[last_two_months, 0]),
    "o-",
    markersize=3,
    color="C0",
    label="Observed history",
)
ax.axvline(setup.t_obs - 0.5, color="black", linestyle="--", linewidth=1)
shade_promotion(ax)
ax.legend(loc="upper left", fontsize=9)
ax.set(xlabel="day", ylabel="units", title="")
pc.viz["figure"].item().suptitle(
    "Forecast Fan for the Planning Horizon", fontsize=18, fontweight="bold"
);

The fan follows the weekly cycle and jumps in the promotion week because the model knows the price path. It widens slowly with the horizon because the level keeps drifting. Next, let's look at $30$ individual scenarios from the optimization set. These paths are what the optimizer sees.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
lines = ax.plot(
    days_future, np.asarray(demand_opt[:30]).T, color="C2", alpha=0.3, linewidth=1
)
lines[0].set_label("Scenario")
ax.plot(
    days_future,
    np.asarray(demand_opt.mean(axis=0)),
    color="C2",
    linewidth=3,
    label="Forecast mean",
)
shade_promotion(ax)
ax.legend(loc="upper left")
ax.set(xlabel="day", ylabel="units")
fig.suptitle("30 Joint Demand Scenarios", fontsize=18, fontweight="bold");

Each path is one joint draw of the $15$ days: the optimizer scores a plan against each of them, and the rule decides how to aggregate the scores. The paths are noisy around the mean, and they share the level and the promotion lift.

Why do we insist on joint draws? The chance constraint and the newsvendor rule need quantiles of the *cumulative* demand $C_t$. If the days were independent, the day-to-day noise would average out and the quantile path would be narrow. The shared level and the shared parameters make the days positively correlated, which widens the quantile of the sum, and only joint draws carry that correlation. Let's check this by shuffling each day of the scenario matrix independently, which keeps every marginal distribution and destroys the correlation, and comparing the $0.95$ quantile of the cumulative demand under both.

In [ ]:
rng_key, key_shuffle = random.split(rng_key)
shuffle_each_day = jax.vmap(random.permutation, in_axes=(0, 1), out_axes=1)
demand_independent = shuffle_each_day(
    random.split(key_shuffle, setup.horizon), demand_opt
)
quantile_joint = jnp.quantile(cumulative_demand_opt, 1 - problem.alpha, axis=0)
quantile_independent = jnp.quantile(
    jnp.cumsum(demand_independent, axis=1), 1 - problem.alpha, axis=0
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    days_future, np.asarray(quantile_joint), "o-", color="C2", label="Joint scenarios"
)
ax.plot(
    days_future,
    np.asarray(quantile_independent),
    "s--",
    color="gray",
    label="Days shuffled independently",
)
ax.legend(loc="upper left")
ax.set(xlabel="day", ylabel="0.95 quantile of cumulative demand")
fig.suptitle("Why Joint Scenarios Matter", fontsize=18, fontweight="bold");

The two quantile paths start together and separate as the days accumulate. By the end of the horizon the joint quantile sits at $1{,}509$ units and the independent one at $1{,}442$. The gap of $67$ units is about $4\%$ of the quantile, and it is the part of the safety stock that a per-day forecast would miss.

## Inventory Simulator

The loss $L(a, D)$ is a recursion over days, so we write it as a `jax.lax.scan` that carries the inventory position. Vectorizing over scenarios is one `jax.vmap` with `in_axes=(None, 0, None)`: the plan is shared, the demand path is batched. The `problem` tuple holds Python floats, so we mark it static under `jax.jit`. A `Scalar` is a float array with no axes.

In [ ]:
Scalar = Float[Array, ""]


def simulate(
    orders: Float[Array, " horizon"], demand: Float[Array, " horizon"], problem: Problem
) -> tuple[Scalar, Float[Array, " horizon"]]:
    """Loss and inventory path of one plan against one demand path."""

    def step(
        inventory: Scalar, inputs: tuple[Scalar, Scalar]
    ) -> tuple[Scalar, tuple[Scalar, Scalar]]:
        # carry: inventory position at the end of the previous day
        # inputs: today's order and today's demand
        order_t, demand_t = inputs
        inventory = inventory + order_t - demand_t
        # holding cost on the positive part, backorder cost on the negative part
        cost_t = problem.holding_cost * jnp.maximum(
            inventory, 0.0
        ) + problem.backorder_cost * jnp.maximum(-inventory, 0.0)
        return inventory, (inventory, cost_t)

    initial_inventory = jnp.asarray(problem.initial_inventory, dtype=orders.dtype)
    _, (inventory_path, cost_path) = jax.lax.scan(
        step, initial_inventory, (orders, demand)
    )
    return cost_path.sum(), inventory_path


# one plan against a batch of demand paths
simulate_scenarios = jax.jit(
    jax.vmap(simulate, in_axes=(None, 0, None)), static_argnames="problem"
)

We also need the tail average. For $S$ equally weighted scenarios, $\text{CVaR}_\alpha$ is the mean of the $k = \alpha S$ largest losses, which `jax.lax.top_k` selects in one call. We keep $k$ as a Python integer so that the shape is static.

In [ ]:
def cvar(losses: Float[Array, " scenarios"], alpha: float) -> Scalar:
    """Mean of the k = alpha * S largest losses, with k an integer."""
    k = round(alpha * losses.shape[0])  # static: top_k needs a Python integer
    worst, _ = jax.lax.top_k(losses, k)  # the k largest losses, no sort needed
    return worst.mean()

Let's test the simulator on the first of the four plans, the mean forecast plan. Against a single demand path the cheapest plan keeps the inventory at zero: cumulative orders equal cumulative demand minus the initial stock, clipped at zero. The same function gives the chance-constrained plan and the newsvendor plan later on, with a quantile path instead of the mean path.

`jax.lax.cummax` keeps the cumulative orders non-decreasing, so every daily order is non-negative. For a non-decreasing target the `cummax` never binds, and the clip at zero binds only when the day one target is below $I_0$, which is not the case here. We keep the mean demand path, since several cells below use it.

In [ ]:
def plan_from_cumulative_target(
    target_cumulative_demand: Float[Array, " horizon"], problem: Problem
) -> Float[Array, " horizon"]:
    """Daily orders that keep cumulative orders at the target cumulative demand."""
    # the initial stock covers the first part of the target, hence the clip at zero
    cumulative_orders = jax.lax.cummax(
        jnp.maximum(target_cumulative_demand - problem.initial_inventory, 0.0), axis=0
    )
    # daily orders are the increments of the cumulative orders
    return jnp.diff(cumulative_orders, prepend=0.0)


mean_demand_opt = demand_opt.mean(axis=0)
orders_mean = plan_from_cumulative_target(cumulative_demand_opt.mean(axis=0), problem)

This plan has zero loss on the mean path by construction. Let's score it on the optimization scenarios, plot the distribution of its loss and mark the expected loss, the value at risk and the conditional value at risk.

In [ ]:
losses_mean_plan, _ = simulate_scenarios(orders_mean, demand_opt, problem)

pc = az.plot_dist(
    xr.Dataset({"loss": (("sample",), np.asarray(losses_mean_plan))}),
    sample_dims=["sample"],
    kind="kde",
    visuals={
        "point_estimate": False,
        "point_estimate_text": False,
        "credible_interval": False,
    },
    figure_kwargs={"figsize": (10, 5)},
)
ax = pc.viz["plot"]["loss"].item()
ax.axvline(
    float(losses_mean_plan.mean()), color="gray", linewidth=2, label="expected loss"
)
ax.axvline(
    float(jnp.quantile(losses_mean_plan, 1 - problem.alpha)),
    color="C1",
    linewidth=2,
    label=VAR_LABEL,
)
ax.axvline(
    float(cvar(losses_mean_plan, problem.alpha)),
    color="C3",
    linewidth=2,
    label=CVAR_LABEL,
)
ax.legend(loc="upper right")
ax.set(xlabel="loss on one scenario", title="")
pc.viz["figure"].item().suptitle(
    "Loss of the Mean Forecast Plan", fontsize=18, fontweight="bold"
);

The loss distribution has the shape of the concept plot above: most scenarios cost little, and a long right tail of high-demand scenarios pushes the tail average far beyond the expected loss. The plan that costs nothing on the mean path has an expected loss of $420$ over the scenarios, a value at risk of $1{,}543$ and a conditional value at risk of $1{,}920$, more than four times the expected loss. We score it on the held-out scenarios in the evaluation section.

## Decision Plans

The mean forecast plan is built. The chance-constrained plan is the same construction with a quantile path, and the two other plans need an optimizer.

### Chance-Constrained Orders

We replace the mean path by the $(1 - \alpha)$ quantile of cumulative demand, computed across the joint scenarios. We take the quantile with `method="inverted_cdf"`, which returns the smallest value with at least a fraction $1 - \alpha$ of the scenarios at or below it, so the plan is the cheapest one that covers demand in at least $95\%$ of the optimization scenarios on every day.

In [ ]:
quantile_path = jnp.quantile(
    cumulative_demand_opt, 1 - problem.alpha, axis=0, method="inverted_cdf"
)
orders_cc = plan_from_cumulative_target(quantile_path, problem)

### Expected Cost Orders

Only the CVaR plan needs an optimizer, and we solve the $\lambda = 0$ case with it as well so that the linear program below checks both solves. The objective $\text{E}[L] + \lambda \, \text{CVaR}_\alpha(L)$ is convex in $a$ (a sum of hinge terms of linear functions, and a tail average of such sums), and the only constraint is $a \geq 0$.

We use projected gradient descent with Adam: after each step we clip the orders at zero, which is the Euclidean projection onto the feasible set. The whole optimization loop is a `jax.lax.scan` inside `jax.jit`, so there is no Python overhead per step, and the same function vectorizes over $\lambda$ with `jax.vmap`.

The objective is piecewise linear, so the gradient is a subgradient and Adam oscillates around the kinks unless the step size shrinks. We use a one-cycle learning rate schedule (warm up, then decay) and chain it with `optax.contrib.reduce_on_plateau`, which halves the step size whenever the objective stops improving. The plateau rule needs the objective value, so we pass it to `optimizer.update`. The solver returns the objective and the effective step size at every step, and we confirm the optimum against a linear program below.

In [ ]:
def objective(
    orders: Float[Array, " horizon"],
    demand: Float[Array, "scenarios horizon"],
    lam: ScalarLike,
    problem: Problem,
) -> Scalar:
    """Expected loss plus lam times the CVaR of the loss over the scenarios."""
    losses, _ = simulate_scenarios(orders, demand, problem)
    return losses.mean() + lam * cvar(losses, problem.alpha)


def solve(
    demand: Float[Array, "scenarios horizon"],
    lam: ScalarLike,
    orders_init: Float[Array, " horizon"],
    problem: Problem,
    n_steps: int = 3_000,
    learning_rate: float = 1.0,
) -> tuple[
    Float[Array, " horizon"], Float[Array, " n_steps"], Float[Array, " n_steps"]
]:
    """Projected Adam on the expected cost plus lam times CVaR, as a lax.scan."""
    # one cycle: warm up to the peak rate over the first 30% of the steps, then decay
    schedule = optax.linear_onecycle_schedule(
        transition_steps=n_steps,
        peak_value=learning_rate,
        pct_start=0.3,
        pct_final=0.85,
        div_factor=2,
        final_div_factor=3,
    )
    optimizer = optax.chain(
        optax.adam(learning_rate=schedule),
        # average the objective over blocks of 10 steps; halve the step size after
        # 50 blocks (500 steps) without a 0.1% improvement, then wait 50 blocks
        optax.contrib.reduce_on_plateau(
            factor=0.5,
            patience=50,
            cooldown=50,
            accumulation_size=10,
            rtol=1e-3,
            min_scale=1e-2,
        ),
    )
    value_and_grad = jax.value_and_grad(objective)

    def step(
        carry: tuple[Float[Array, " horizon"], optax.OptState], _: None
    ) -> tuple[tuple[Float[Array, " horizon"], optax.OptState], tuple[Scalar, Scalar]]:
        orders, opt_state = carry
        value, grads = value_and_grad(orders, demand, lam, problem)
        # the plateau rule reads the objective, so we pass it as a keyword
        updates, opt_state = optimizer.update(grads, opt_state, orders, value=value)
        orders = jnp.maximum(orders + updates, 0.0)  # projection onto a >= 0
        scale = opt_state[1].scale  # current multiplier of the plateau rule
        return (orders, opt_state), (value, scale)

    (orders, _), (values, scales) = jax.lax.scan(
        step, (orders_init, optimizer.init(orders_init)), None, length=n_steps
    )
    step_sizes = jax.vmap(schedule)(jnp.arange(n_steps)) * scales
    return orders, values, step_sizes


solve_jit = jax.jit(solve, static_argnames=("problem", "n_steps"))

We start from the mean forecast plan and set $\lambda = 0$. We also build the newsvendor plan in closed form and compare the two. For $S$ scenarios the sample average is minimized by the smallest cumulative demand with at least a fraction $p / (p + h)$ of the scenarios at or below it, the $910$-th smallest value for $S = 1{,}000$, which is again what `method="inverted_cdf"` returns.

In [ ]:
orders_saa, trace_saa, step_sizes_saa = solve_jit(demand_opt, 0.0, orders_mean, problem)

newsvendor_path = jnp.quantile(
    cumulative_demand_opt, critical_fractile, axis=0, method="inverted_cdf"
)
orders_newsvendor = plan_from_cumulative_target(newsvendor_path, problem)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    days_future, np.asarray(orders_saa), "o-", color="C2", label="Projected Adam (JAX)"
)
ax.plot(
    days_future,
    np.asarray(orders_newsvendor),
    "s--",
    color="black",
    label="Newsvendor closed form",
)
shade_promotion(ax)
ax.legend(loc="upper left")
ax.set(xlabel="day", ylabel="units ordered")
fig.suptitle(
    "Expected Cost Plan Against the Newsvendor Closed Form",
    fontsize=18,
    fontweight="bold",
);

The two plans lie on top of each other, with a largest difference of $1.6$ units, so the optimizer and the closed form agree. The expected cost plan is a quantile plan at the critical fractile, and the chance-constrained plan is the same construction at $0.95$.

### CVaR Orders

Now we put a price on the tail with $\lambda = 1$: one unit of average loss in the worst $5\%$ of scenarios counts as much as one unit of expected loss.

In [ ]:
orders_cvar, trace_cvar, step_sizes_cvar = solve_jit(
    demand_opt, 1.0, orders_mean, problem
)

Let's look at the optimizer traces for both solves: the objective on top and the effective step size, the schedule times the plateau multiplier, below it.

In [ ]:
steps = np.arange(1, trace_saa.shape[0] + 1)

fig, axes = plt.subplots(nrows=2, figsize=(10, 7), sharex=True, layout="constrained")
ax = axes[0]
ax.plot(
    steps, np.asarray(trace_saa), color="C2", label=r"$\lambda = 0$ (expected cost)"
)
ax.plot(
    steps,
    np.asarray(trace_cvar),
    color="C3",
    label=r"$\lambda = 1$ (expected cost + CVaR)",
)
ax.legend()
ax.set(ylabel="objective", xscale="log")
ax = axes[1]
ax.plot(steps, np.asarray(step_sizes_saa), color="C2")
ax.plot(steps, np.asarray(step_sizes_cvar), color="C3")
ax.set(xlabel="step", ylabel="effective step size", yscale="log")
fig.suptitle("Optimizer Traces", fontsize=18, fontweight="bold");

Both objectives drop within the first $50$ steps and stay flat for the rest of the run, at $185$ for the expected cost and $625$ for the expected cost plus CVaR. The step size follows the one-cycle schedule up to its peak at step $900$. From there the plateau rule halves it three times, because the objective no longer improves, and the final step size of $0.02$ units lets the iterates settle.

We now have the four plans, so let's collect them in a list of `Plan` tuples and compare them. The shaded column is again the promotion week.

In [ ]:
class Plan(NamedTuple):
    """One order plan with its name."""

    name: str
    orders: Float[Array, " horizon"]


plans = [
    Plan("Mean forecast", orders_mean),
    Plan("Chance constrained", orders_cc),
    Plan("Expected cost", orders_saa),
    Plan("CVaR", orders_cvar),
]
plan_names = [plan.name for plan in plans]
orders_long = pl.DataFrame(
    {
        "plan": np.repeat(plan_names, setup.horizon),
        "day": np.tile(days_future, len(plans)),
        "orders": np.concatenate([np.asarray(plan.orders) for plan in plans]),
    }
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=orders_long, x="day", y="orders", hue="plan", marker="o", ax=ax)
ax.plot(
    days_future,
    np.asarray(mean_demand_opt),
    color="black",
    linestyle="--",
    label="Mean demand",
)
shade_promotion(ax)
ax.legend(loc="upper left", fontsize=9)
ax.set(xlabel="day", ylabel="units ordered")
fig.suptitle("Order Plans", fontsize=18, fontweight="bold");

The mean forecast plan follows the mean demand, with a first order of $22$ units because the $50$ units of initial stock cover most of day one. The three other plans order a buffer above the mean from the first day on, and every plan orders more in the promotion week.

The buffer is large because every plan is fixed in advance, so it has to hedge the uncertainty of cumulative demand over the whole horizon, and because the holding cost is one tenth of the backorder cost.

The CVaR plan orders the most on most days from day $3$ on, and from day $6$ its cumulative orders are the largest of the four, so it carries the largest buffer through the promotion week. The tail objective is nearly flat along shifts of units between neighboring days, so the solver stops anywhere on that flat region, which is why its path is less smooth than the two quantile plans.

## Out of Sample Evaluation

We score every plan on the $3{,}000$ held-out scenarios. Besides the expected loss and the tail average, we report the stockout probability per day (the fraction of scenario and day pairs with a backlog, which is the average over days of the per-day probability) and the probability of at least one stockout in the horizon. We simulate each plan once and keep the losses and the inventory paths for the figures below.

In [ ]:
def summarize(
    losses: Float[Array, " scenarios"],
    inventory: Float[Array, "scenarios horizon"],
    alpha: float,
) -> dict[str, float]:
    """Expected loss, CVaR and stockout frequencies of one plan over the scenarios."""
    return {
        "expected loss": float(losses.mean()),
        f"CVaR of loss (worst {alpha:.0%})": float(cvar(losses, alpha)),
        "stockout probability per day": float((inventory < 0).mean()),
        "probability of a stockout in the horizon": float(
            (inventory < 0).any(axis=1).mean()
        ),
    }


scores = {
    plan.name: simulate_scenarios(plan.orders, demand_eval, problem) for plan in plans
}
results = pl.DataFrame(
    [
        {"plan": name, **summarize(*score, problem.alpha)}
        for name, score in scores.items()
    ]
)
metrics = [column for column in results.columns if column != "plan"]
plan_labels = [name.replace(" ", "\n") for name in plan_names]

fig, axes = plt.subplots(
    nrows=2, ncols=2, figsize=(12, 8), sharex=True, layout="constrained"
)

for ax, metric in zip(axes.flat, metrics, strict=True):
    sns.barplot(data=results, x="plan", y=metric, hue="plan", legend=False, ax=ax)
    ax.set(title=metric, xlabel="", ylabel="")

for ax in axes[-1]:
    ax.set_xticks(range(len(plans)), plan_labels)

fig.suptitle("Held-Out Scores by Plan", fontsize=18, fontweight="bold");

Here are three important remarks.

- **The point forecast is expensive.** The mean forecast plan has the worst expected loss ($449$) and the worst tail ($2{,}085$), and it runs out of stock on half of the days. Its zero cost on the mean path told us nothing about its cost on the distribution.
- **The expected cost plan lands on the critical fractile.** Its stockout probability per day is $9.7\%$, close to $1 - p / (p + h) = 9.1\%$. That is the risk-neutral answer, and it follows from the costs alone.
- **The chance constraint and CVaR buy reliability differently.** The chance-constrained plan lands at $5.2\%$ per day on the held-out scenarios, close to its $5\%$ target. The CVaR plan with $\lambda = 1$ pushes the stockout probability per day down to $4.7\%$ and the tail loss down to $489$, and pays for it with the highest expected loss of the three ($213$ against $197$ and $190$). The lower right panel reverses the per-day comparison: $20.9\%$ of the CVaR horizons and $18.3\%$ of the chance-constrained horizons see at least one stockout.

The chance constraint holds day by day, so let's look at the stockout probability on each day of the horizon. The dashed line is the target $\alpha = 0.05$.

In [ ]:
stockout_long = pl.DataFrame(
    {
        "plan": np.repeat(plan_names, setup.horizon),
        "day": np.tile(days_future, len(plans)),
        "stockout probability": np.concatenate(
            [
                np.asarray((inventory < 0).mean(axis=0))
                for _, inventory in scores.values()
            ]
        ),
    }
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=stockout_long, x="day", y="stockout probability", hue="plan", marker="o", ax=ax
)
ax.axhline(problem.alpha, color="black", linestyle="--", label=r"target $\alpha$")
shade_promotion(ax)
ax.legend(loc="upper left", fontsize=9)
ax.set(xlabel="day", ylabel="stockout probability")
fig.suptitle("Stockout Probability by Day", fontsize=18, fontweight="bold");

The chance-constrained plan stays between $4\%$ and $6\%$ on every day, the expected cost plan sits between $8\%$ and $11\%$ on every day, and the mean forecast plan runs out in half of the scenarios every day. The CVaR plan is different: it runs out in $11\%$ of the scenarios on day one and between $2.5\%$ and $4.2\%$ from day six on.

The tail objective cares about the high-demand horizons, and those separate from the rest only after several days, so it adds stock late and not early. This is also why the CVaR plan has more horizons with at least one stockout than the chance-constrained plan: its stockouts cluster on the first days.

Let's look at the loss distributions behind these numbers. We leave out the mean forecast plan, whose losses run far to the right of the other three, and we clip the horizontal axis at the $99$th percentile of the pooled losses to see the bulk of the distributions.

In [ ]:
losses_by_plan = {
    name: xr.Dataset({"loss": (("sample",), np.asarray(losses))})
    for name, (losses, _) in scores.items()
    if name != "Mean forecast"
}
loss_clip = float(
    np.quantile(
        np.concatenate([ds["loss"].values for ds in losses_by_plan.values()]), 0.99
    )
)

pc = az.plot_dist(
    losses_by_plan,
    sample_dims=["sample"],
    kind="kde",
    color=["C1", "C2", "C3"],  # same colors as in the other figures
    visuals={"point_estimate_text": False},
    figure_kwargs={"figsize": (10, 5)},
)
pc.add_legend("model", title="plan")
pc.viz["plot"]["loss"].item().set(
    xlim=(0, loss_clip), xlabel="loss on one scenario", title=""
)
pc.viz["figure"].item().suptitle(
    "Loss Distributions on Held-Out Scenarios", fontsize=18, fontweight="bold"
);

The expected cost plan has the most mass at low losses, and the bar chart above showed that it also has the highest tail average of the three. Its largest held-out loss is about $3{,}000$, as the scatter plot below shows. The CVaR plan trades some of that low-loss mass for a shorter tail, with a largest loss of about $2{,}300$, and the chance-constrained plan sits in between. The dots and bars at the bottom are the means and $94\%$ intervals.

Why are the right tails so much longer than the left? The loss is bounded below by zero, and the two arms of the loss have different slopes.

One unit of demand less than planned on day $t$ stays on the shelf for the remaining $H - t + 1$ days and costs about $h \, (H - t + 1)$ in holding. One unit more than planned stays in backlog for the same days and costs about $p \, (H - t + 1)$. Both effects persist for the same number of days, so persistence is symmetric and the asymmetry comes from the cost ratio $p / h = 10$.

The plans sit above the mean demand, so most scenarios land on the cheap holding arm and a minority lands on the expensive backorder arm. A second source is the right skew of the scenarios themselves, because the level enters the rate through an exponential.

Let's confirm this by plotting the loss of every held-out scenario against its total demand over the horizon.

In [ ]:
scatter_long = pl.DataFrame(
    {
        "plan": np.repeat(plan_names, demand_eval.shape[0]),
        "total demand": np.tile(np.asarray(demand_eval.sum(axis=1)), len(plans)),
        "loss": np.concatenate([np.asarray(losses) for losses, _ in scores.values()]),
    }
)

fig, ax = plt.subplots()
sns.scatterplot(
    data=scatter_long, x="total demand", y="loss", hue="plan", s=8, alpha=0.3, ax=ax
)
ax.legend(loc="upper left")
ax.set(xlabel="total demand over the horizon (held-out scenario)", ylabel="loss")
fig.suptitle("Loss Against Total Demand", fontsize=18, fontweight="bold");

Each plan traces a check mark: a shallow left arm where we pay holding costs and a steep right arm where we pay backorder costs. The buffered plans move the bottom of the check mark to the right, so fewer scenarios reach the steep arm. The mean forecast plan has its bottom at the mean demand, and half of the scenarios climb the steep arm.

Next, let's look at the inventory paths under each plan with `az.plot_lm`, one panel per plan. The black dots are the inventory path under the true future demand, which we return to at the end.

In [ ]:
realized_paths = {
    plan.name: simulate(plan.orders, demand_future_true, problem) for plan in plans
}
inventory_tree = az.from_dict(
    {
        "posterior_predictive": {
            "inventory": np.stack(
                [np.asarray(inventory) for _, inventory in scores.values()], axis=1
            )[None]
        },
        "observed_data": {
            "inventory": np.stack(
                [np.asarray(inventory) for _, inventory in realized_paths.values()]
            )
        },
        "constant_data": {"day": np.tile(days_future.astype(float), (len(plans), 1))},
    },
    coords={"time": days_future, "plan": plan_names},
    dims={"inventory": ["plan", "time"], "day": ["plan", "time"]},
)

pc = az.plot_lm(
    inventory_tree,
    x="day",
    y="inventory",
    y_obs="inventory",
    plot_dim="time",
    smooth=False,
    ci_prob=(0.5, HDI_PROB),
    point_estimate="mean",
    aes={"color": ["plan"]},
    aes_by_visuals={"pe_line": ["color"]},
    color=[f"C{i}" for i in range(len(plans))],
    visuals={
        "pe_line": {"alpha": 1.0},
        "observed_scatter": {"color": "black", "alpha": 1.0, "s": 14},
        "xlabel": False,
        "ylabel": False,
    },
    col_wrap=2,
    figure_kwargs={"figsize": (14, 8), "sharex": True, "sharey": True},
)
lighten_bands(pc, "day")

for name in plan_names:
    ax = pc.get_target("day", {"plan": name})
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
    ax.set(title=name)

# one legend for all panels, read from the first one
first = {"plan": plan_names[0]}
pc.viz["ci_band"]["day"].sel(prob=HDI_PROB, **first).item().set_label(
    f"{HDI_PROB:.0%} HDI"
)
pc.viz["ci_band"]["day"].sel(prob=0.5, **first).item().set_label("50% HDI")
pc.viz["pe_line"]["day"].sel(**first).item().set_label("Mean")
pc.viz["observed_scatter"]["day"].sel(**first).item().set_label(
    "Path under the true future"
)
pc.get_target("day", first).legend(loc="upper left", fontsize=9)

fig = pc.viz["figure"].item()
fig.supxlabel("day")
fig.supylabel("inventory position")
fig.suptitle("Inventory Position Under Each Plan", fontsize=18, fontweight="bold");

The bands are the $50\%$ and $94\%$ HDIs of the inventory position over the held-out scenarios, and the line is the mean. The mean forecast plan keeps the expected inventory at zero, so half of its band sits below the dashed line. The other plans lift the whole band.

The CVaR plan keeps the lower edge of the $94\%$ interval at about zero and above zero from day $11$ on, the chance-constrained plan lets it fall to about $-35$ at the end of the horizon, and the expected cost plan accepts a lower edge near $-80$.

Note that the mean position grows over the horizon under all three. The uncertainty of cumulative demand grows with the horizon, and a plan that cannot react has to hedge all of it up front. A plan with recourse, which re-forecasts and re-orders every day, would hedge one day at a time. We list this rolling horizon in the next steps.

## Pareto Frontier

The weight $\lambda$ has no natural scale. The useful object is the frontier it traces: for each $\lambda$ we solve the problem, score the plan out of sample, and plot the expected loss against the tail loss and against the stockout frequency. Because `solve` is a pure function, the whole sweep is one `jax.vmap` over the vector of weights, compiled once.

We also record which scenarios make up the worst $5\%$ under each plan. The share of them with a total demand below the median tells us whether the tail is made of high-demand or of low-demand scenarios.

In [ ]:
def low_demand_tail_share(
    losses: Float[Array, " scenarios"],
    demand: Float[Array, "scenarios horizon"],
    alpha: float,
) -> float:
    """Share of the worst alpha fraction of scenarios with below-median total demand."""
    # indices of the k scenarios with the largest losses
    _, worst = jax.lax.top_k(losses, round(alpha * losses.shape[0]))
    total_demand = demand.sum(axis=1)
    return float((total_demand[worst] < jnp.median(total_demand)).mean())

Let's run the sweep and score every plan on the held-out scenarios, including the tail composition.

In [ ]:
lambdas = jnp.array([0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 8.0])
solve_frontier = jax.jit(
    jax.vmap(solve, in_axes=(None, 0, None, None)),
    static_argnames=("problem", "n_steps"),
)
orders_frontier, _, _ = solve_frontier(demand_opt, lambdas, orders_mean, problem)

frontier_rows = []

for lam, orders in zip(lambdas, orders_frontier, strict=True):
    losses, inventory = simulate_scenarios(orders, demand_eval, problem)
    frontier_rows.append(
        {
            "lambda": float(lam),
            **summarize(losses, inventory, problem.alpha),
            "low-demand share of the tail": low_demand_tail_share(
                losses, demand_eval, problem.alpha
            ),
        }
    )

frontier = pl.DataFrame(frontier_rows)

First, let's look at the composition of the tail as a function of $\lambda$.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    np.arange(len(lambdas)),
    frontier["low-demand share of the tail"].to_numpy(),
    "o-",
    color="black",
)
ax.set(
    xticks=np.arange(len(lambdas)),
    xticklabels=[f"{lam:g}" for lam in frontier["lambda"]],
    xlabel=r"$\lambda$",
    ylabel=f"low-demand share of the {TAIL_LABEL}",
    ylim=(0, 1),
)
fig.suptitle("Composition of the Tail", fontsize=18, fontweight="bold");

At $\lambda = 0$ the worst $5\%$ of the held-out scenarios are all high-demand horizons: none of them has a below-median total demand. As $\lambda$ grows the plan carries more stock, and the tail fills with low-demand horizons that pay holding cost on the whole buffer, up to $68\%$ of the tail at $\lambda = 2$ and beyond. Next, let's plot the frontier and mark the three plans on it. The mean forecast plan is far outside both panels, so we leave it out.

In [ ]:
frontier_metrics = [metrics[1], metrics[2]]  # tail average and stockout probability

fig, axes = plt.subplots(nrows=2, figsize=(10, 10), sharex=True, layout="constrained")

for ax, metric in zip(axes, frontier_metrics, strict=True):
    ax.plot(
        frontier["expected loss"],
        frontier[metric],
        "o-",
        color="black",
        label=r"CVaR frontier over $\lambda$",
    )

    for j, (lam, x_val, y_val) in enumerate(
        zip(
            frontier["lambda"], frontier["expected loss"], frontier[metric], strict=True
        )
    ):
        ax.annotate(
            rf"$\lambda={lam:g}$",
            (x_val, y_val),
            textcoords="offset points",
            xytext=(6, 6) if j % 2 == 0 else (-6, -12),
            ha="left" if j % 2 == 0 else "right",
            fontsize=9,
            color="black",
        )

    for i, row in enumerate(results.iter_rows(named=True)):
        if row["plan"] == "Mean forecast":
            continue

        ax.scatter(
            row["expected loss"],
            row[metric],
            s=120,
            color=f"C{i}",
            edgecolor="black",
            zorder=5,
            label=row["plan"],
        )

    ax.set(ylabel=metric)
    ax.margins(x=0.08)

axes[0].legend(loc="upper right", fontsize=9)
axes[-1].set(xlabel="expected loss")
fig.suptitle("Cost Versus Risk on Held-Out Scenarios", fontsize=18, fontweight="bold");

The frontier is steep on the left and flat on the right. A small $\lambda$ removes a large part of the tail loss for a modest increase in expected loss: $\lambda = 0.1$ cuts the tail loss by $29\%$ for a $2.5\%$ increase in expected loss. Beyond $\lambda \approx 1$ the tail loss stops improving while the expected loss keeps rising, and in the lower panel the stockout frequency stops falling at about $4.8\%$. The tail composition explains why: at large $\lambda$ the CVaR term shapes the plan around the low-demand horizons as well, and the scenarios in the middle run out as often as before.

This is the curve to put in front of the business: it prices one unit of tail loss, or one percentage point of stockout probability, in units of expected loss.

Here is an important observation. The chance-constrained plan sits close to the frontier in the upper panel and below it in the lower panel. Both statements are consistent. CVaR trades expected loss against the tail average, and the chance constraint targets the stockout frequency. Each is efficient for the metric it targets, so the requirement decides which one to use.

## Convex Solver Cross Check

The projected gradient solver is convenient because it stays in JAX, but it is a first-order method on a non-smooth objective, so let's confirm that it reaches the optimum. Let $z$ be a loss threshold. The expected cost plus CVaR objective is a linear program once we use the Rockafellar and Uryasev representation

$$
\text{CVaR}_\alpha(L) = \min_{z} \left\{ z + \frac{1}{\alpha} \, \text{E}\left[(L - z)^{+}\right] \right\},
$$

where the minimizing $z$ is the value at risk. The function inside the braces is easy to picture. Let's evaluate it on the losses of the expected cost plan over the optimization scenarios, for a grid of thresholds, and compare its minimum with `cvar`.

In [ ]:
losses_saa_opt, _ = simulate_scenarios(orders_saa, demand_opt, problem)
threshold_grid = np.linspace(0, float(losses_saa_opt.max()), 400)
# z + E[(L - z)^+] / alpha on the grid, with the expectation as a scenario average
ru_objective = (
    threshold_grid
    + np.mean(
        np.maximum(np.asarray(losses_saa_opt)[None, :] - threshold_grid[:, None], 0.0),
        axis=1,
    )
    / problem.alpha
)
tail_size = round(problem.alpha * N_SCENARIOS_OPT)  # k = alpha * S

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    threshold_grid,
    ru_objective,
    color="C0",
    linewidth=2,
    label=r"$z + \text{E}[(L - z)^+] / \alpha$",
)
ax.axhline(
    float(cvar(losses_saa_opt, problem.alpha)),
    color="C3",
    linewidth=2,
    label=f"{CVAR_LABEL} (mean of the {tail_size} largest losses)",
)
ax.axvline(
    float(jnp.quantile(losses_saa_opt, 1 - problem.alpha)),
    color="C1",
    linewidth=2,
    label=VAR_LABEL,
)
ax.legend(loc="upper right")
ax.set(
    xlabel="threshold $z$",
    ylabel="value",
    ylim=(0, 3 * float(cvar(losses_saa_opt, problem.alpha))),
)
fig.suptitle(
    "The Rockafellar and Uryasev Representation", fontsize=18, fontweight="bold"
);

The curve is convex in $z$, its minimum sits at the value at risk, and the minimum value is the conditional value at risk that `cvar` computes with `top_k`. The linear program minimizes over $z$ and over the plan at the same time.

For $S$ equally weighted scenarios and an integer $k = \alpha S$, this minimum equals the mean of the $k$ largest losses, so the linear program minimizes the same objective as `solve`. With $L_s = L(a, D^{(s)})$, one auxiliary variable $u_s \geq (L_s - z)^{+}$ per scenario and the hinge terms of $L$ written with `cp.pos`, cvxpy hands the problem to HiGHS.

Building the expression tree over $1{,}000$ scenarios sometimes raises a numpy `RuntimeWarning` (`overflow` or `invalid value encountered in reduce`). It comes from a shape check inside cvxpy that sums an uninitialized array, so it does not affect the solution, and we silence that one message around those lines.

In [ ]:
def solve_lp(
    demand: Float[Array, "scenarios horizon"], lam: float, problem: Problem
) -> Float[Array, " horizon"]:
    """The same objective as `solve`, as a linear program."""
    demand = np.asarray(demand, dtype=np.float64)
    num_scenarios, horizon = demand.shape
    cumulative_demand = np.cumsum(demand, axis=1)
    orders = cp.Variable(horizon, nonneg=True)  # the plan a
    var = cp.Variable()  # the threshold z
    excess = cp.Variable(num_scenarios, nonneg=True)  # u_s >= (L_s - z)^+

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=".*encountered in reduce",
            category=RuntimeWarning,
        )
        # inventory of every scenario on every day, I_t = I_0 + A_t - C_t
        inventory = (
            problem.initial_inventory
            + cp.reshape(cp.cumsum(orders), (1, horizon), order="C")
            - cumulative_demand
        )
        # loss of every scenario: the two hinge terms summed over days
        losses = cp.sum(
            problem.holding_cost * cp.pos(inventory)
            + problem.backorder_cost * cp.pos(-inventory),
            axis=1,
        )
        cvar_term = var + cp.sum(excess) / (problem.alpha * num_scenarios)
        lp = cp.Problem(
            cp.Minimize(cp.sum(losses) / num_scenarios + lam * cvar_term),
            [excess >= losses - var],
        )

    lp.solve(solver=cp.HIGHS)
    return jnp.asarray(orders.value, dtype=jnp.float32)

We solve the two problems with cvxpy and overlay the order plans of both solvers. We evaluate `objective` at both solutions, so that the two values come from the same function, and the panel titles carry the relative gap and the largest difference between the two order vectors.

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(14, 5), sharey=True, layout="constrained")

for ax, (name, lam, orders_jax) in zip(
    axes, [("Expected cost", 0.0, orders_saa), ("CVaR", 1.0, orders_cvar)], strict=True
):
    orders_lp = solve_lp(demand_opt, lam, problem)
    value_jax = float(objective(orders_jax, demand_opt, lam, problem))
    value_lp = float(objective(orders_lp, demand_opt, lam, problem))
    relative_gap = value_jax / value_lp - 1
    max_order_diff = float(jnp.abs(orders_jax - orders_lp).max())
    ax.plot(
        days_future,
        np.asarray(orders_jax),
        "o-",
        color="C0",
        label="Projected Adam (JAX)",
    )
    ax.plot(
        days_future,
        np.asarray(orders_lp),
        "s--",
        color="C1",
        label="Linear program (cvxpy)",
    )
    shade_promotion(ax)
    ax.legend(loc="upper left")
    ax.set(
        xlabel="day",
        title=f"{name}\nobjective gap {relative_gap:.1e}, "
        f"max order difference {max_order_diff:.1f} units",
    )

axes[0].set(ylabel="units ordered")
fig.suptitle("JAX Solver Against the Linear Program", fontsize=18, fontweight="bold");

The two solvers agree! The relative objective gaps are $2 \times 10^{-6}$ for the expected cost plan and $3 \times 10^{-5}$ for the CVaR plan. The order quantities differ by up to $1.6$ units in the first case and up to $4.5$ units in the second, on orders of about $100$ units. This is the flat region of a piecewise-linear objective near its optimum: shifting units between neighboring days barely moves the objective, and the CVaR objective is flatter because only $50$ scenarios enter the tail term. We use the JAX solver to vectorize over weights or over products, and the LP when the problem gains integer decisions or coupling constraints.

Finally, as a single sanity check, we score the four plans on the one true future path we held out at the start. One path is noisy, so we read this as a consistency check. We show the realized loss and the number of days in backlog.

In [ ]:
realized = pl.DataFrame(
    [
        {
            "plan": name,
            "realized loss": float(loss),
            "days in backlog": int((inventory < 0).sum()),
        }
        for name, (loss, inventory) in realized_paths.items()
    ]
)

fig, axes = plt.subplots(ncols=2, figsize=(12, 5), layout="constrained")

for ax, metric in zip(axes, ["realized loss", "days in backlog"], strict=True):
    sns.barplot(data=realized, x="plan", y=metric, hue="plan", legend=False, ax=ax)
    ax.set_xticks(range(len(plans)), plan_labels)
    ax.set(title=metric, xlabel="", ylabel="")

fig.suptitle("Realized Loss on the Held-Out Future", fontsize=18, fontweight="bold");

The true path ran above the forecast mean on $8$ of the $15$ days and by about $3\%$ in total. The mean forecast plan was in backlog on $11$ days, from day one, where the true demand of $94$ units met an order of $22$ units plus the initial stock, and through the second week, so it paid the most. The three buffered plans never ran out, and the expected cost plan paid the least of them because it held the smallest buffer. On the $3{,}000$ held-out scenarios the CVaR plan has the best tail and the chance-constrained plan the best service level, so one path does not rank the plans.

## Conclusion

### Key Findings

- **The forecast is the scenario matrix.** All four plans are built from the same $1{,}000$ joint posterior predictive scenarios and scored on the same $3{,}000$ held-out ones. The only modeling choice at the decision step is the rule applied to the loss distribution, and that choice moved the expected loss by a factor of more than two across plans.
- **A point forecast hides the cost.** The mean forecast plan costs nothing on the mean path and the most on the distribution. The loss is non-linear in demand, so the expectation of the loss differs from the loss at the expected demand.
- **The risk-neutral plan is the critical fractile.** Minimizing expected cost gives a stockout probability per day near $1 - p / (p + h)$. A service level target needs a chance constraint. A CVaR term prices the tail loss and does not control the frequency.
- **Each rule is efficient for its own metric.** The chance-constrained plan sits on the CVaR frontier when we measure tail loss and beats it when we measure stockout frequency, because frequency is what it targets. A large CVaR weight stops improving the stockout frequency, because the tail fills with low-demand horizons that pay holding cost. Pick the rule that matches the requirement the business states.
- **Known covariates sharpen the scenarios.** The price path is set by the retailer, so the forecast knows the promotion week in the horizon and every plan raises its orders for that week.
- **JAX covers the whole pipeline.** The loss is a `lax.scan`, the scenarios are a `vmap`, the tail is a `lax.top_k`, the solver is a `lax.scan` under `jit`, and the frontier is a `vmap` over the weight. The projected gradient solver matched the LP optimum to $3 \times 10^{-5}$ or better.

### Model Limitations

- **No recourse.** All orders are fixed at the start of the horizon. A rolling horizon that re-forecasts and re-solves each day, and commits only the next order, is the practical version of this problem.
- **No lead time.** Orders arrive on the day we planned them for. The VN2 setting has a two-week delivery lead time, which moves the decision to the inventory projected at the delivery date.
- **Same model in and out of sample.** We evaluate the plans on scenarios from the same model that produced the optimization set, so model misspecification does not show up in the held-out scores. A backtest on real data would expose it.
- **Linear backorders.** Backorders carry at a linear cost. Lost sales, order minimums and fixed ordering costs change the simulator, and integer order quantities change the solver.
- **In-sample optimism.** The sample average approximation with $1{,}000$ scenarios is optimistic in sample. We scored out of sample for that reason.

### Recommendations

1. **Report the whole frontier.** The weight $\lambda$ is a business decision, and the frontier shows what one percentage point of reliability costs.
2. **Keep the scenarios joint.** Marginal quantiles per day break every constraint that couples days, products or hierarchy levels.
3. **Validate the first-order solver once.** Check it against a convex solver once per problem class, then use the JAX version for the sweeps.

## Next Steps

1. **Rolling horizon.** Wrap the solve in `numpyro_forecast.backtest` so that each window re-forecasts, re-solves and commits one order.
2. **Lead time and lost sales.** Add a pipeline of in-transit orders to the scan carry, as in the VN2 setting, and replace the backlog by a lost-sales branch. Only the simulator changes.
3. **Many products.** Fit a hierarchical forecast and `vmap` the solver over the product axis, with a shared budget constraint as a second projection step.
4. **Distributionally robust variant.** Reweight the scenarios adversarially within a divergence ball to see how much of the frontier survives when the forecast tails are not trusted.

## References

- Szabłowski, B. (2026). [One Global Model, Many Behaviors: Stockout-Aware Feature Engineering and Dynamic Scaling for Multi-Horizon Retail Demand Forecasting with a Cost-Aware Ordering Policy (VN2 Winner Report)](https://arxiv.org/abs/2601.18919). arXiv:2601.18919.
- Saunders, D., Viera, B., Laramore, J. and Wimpey, E. (2026). [Designing Optimal Energy Generation Plans with PyMC and FICO Xpress](https://www.pymc-labs.com/blog-posts/probabilistic-forecasting-optimization-under-uncertainty). PyMC Labs blog.
- Rockafellar, R. T. and Uryasev, S. (2000). [Optimization of conditional value-at-risk](https://www.risk.net/journal-risk/2161159/optimization-conditional-value-risk). *Journal of Risk*, 2(3), 21-41.
- Shapiro, A., Dentcheva, D. and Ruszczyński, A. (2021). [Lectures on Stochastic Programming: Modeling and Theory](https://doi.org/10.1137/1.9781611976595), third edition. SIAM.
- Birge, J. R. and Louveaux, F. (2011). [Introduction to Stochastic Programming](https://link.springer.com/book/10.1007/978-1-4614-0237-4), second edition. Springer.
- FICO Xpress Community. [Stochastic Energy Planning: PyMC + FICO Xpress Optimization](https://github.com/fico-xpress/xpress-community/tree/main/StochasticEnergyPlanning).
- Orduz, J. [NumPyro Forecast documentation](https://juanitorduz.github.io/numpyro_forecast/).
- Diamond, S. and Boyd, S. (2016). [CVXPY: A Python-embedded modeling language for convex optimization](https://jmlr.org/papers/v17/15-408.html). *Journal of Machine Learning Research*, 17(83), 1-5.
- DeepMind. [Optax documentation](https://optax.readthedocs.io/).